<a href="https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/colab/vit_optimizer_diagnostics/vit_optimizer_dynamics_geometry_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ViT Optimizer Diagnostics Lab

CIFAR-10의 작은 ViT를 같은 초기값에서 **SGD / AdamW / Prodigy / Muon**으로 50 epoch 학습합니다.

실험 흐름:

```text
성능
→ gradient / update dynamics
→ representation geometry
→ class geometry
→ function-space 비교
→ Hessian spectrum
→ mode connectivity / basin
```

기존의 중요한 측정은 유지하고, 더 포괄적인 측정은 상위 진단으로 확장했습니다.

- Hessian top eigenvalue → Lanczos Ritz spectrum
- 단순 weight interpolation → barrier height + midpoint averaging
- gradient norm → gradient cosine / noise / update cosine / displacement까지 확장
- PCA/UMAP 중심 → covariance spectrum / CKA / probe / Neural Collapse / margin / manifold 통계 중심

이번 버전에서는 HP sweep과 scaling sweep은 제외합니다.


## 0. 환경 설정

T4 실행 시간을 고려해 다음을 기본값으로 둡니다.

- epoch: 50
- batch size: 512
- DataLoader workers: 4
- AMP: GPU 사용 시 활성화
- representation checkpoint: 0, 10, 25, 50 epoch
- Hessian: 초기점과 최종점에서만 계산


In [ ]:
!pip -q install datasets prodigyopt tensorboard scikit-learn
!git clone -q https://github.com/HisameOgasahara/deep-learning-diagnostics-and-improvement.git /content/dldi || git -C /content/dldi pull -q

import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from google.colab import drive
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

LAB_DIR = "/content/dldi/colab/vit_optimizer_diagnostics"
sys.path.insert(0, LAB_DIR)

from vit_lab_model_optim import SmallViT
from vit_lab_train import train_one_optimizer
from vit_lab_repr import extract_features_and_logits, representation_diagnostics
from vit_lab_landscape import (
    run_function_space_comparison,
    run_hessian_diagnostics,
    run_mode_connectivity,
)


### 실험 설정


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 7

EPOCHS = 50
BATCH_SIZE = 512
NUM_WORKERS = 4

OPTIMIZER_NAMES = ["sgd", "adamw", "prodigy", "muon"]

DIAG_EPOCHS = [0, 10, 25, 50]
DYNAMICS_EVERY = 10

REP_TRAIN_SAMPLES = 5000
REP_VAL_SAMPLES = 2000
CONNECTIVITY_SAMPLES = 1000

HESSIAN_BATCH_SIZE = 64
LANCZOS_STEPS = 16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Native Muon available:", hasattr(torch.optim, "Muon"))


### 결과 저장 위치


In [ ]:
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/deep_learning_diagnostics_vit")
CSV_DIR = DRIVE_ROOT / "csv"
FIG_DIR = DRIVE_ROOT / "figures"
TB_DIR = DRIVE_ROOT / "tensorboard"
SUMMARY_DIR = DRIVE_ROOT / "summaries"
LOCAL_CKPT_DIR = Path("/content/vit_optimizer_checkpoints")

for folder in [CSV_DIR, FIG_DIR, TB_DIR, SUMMARY_DIR, LOCAL_CKPT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


## 1. 데이터 준비

학습 입력에는 crop / horizontal flip을 적용합니다.

CKA, covariance spectrum, probe처럼 checkpoint 사이 representation을 비교하는 진단에는 **augmentation이 없는 고정 입력**을 사용합니다.


In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

hf_train = load_dataset("uoft-cs/cifar10", split="train")


In [ ]:
class CIFAR10FromHF(Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        row = self.dataset[index]
        image = row["img"].convert("RGB")
        label = int(row["label"])
        return self.transform(image), label


train_augmented = CIFAR10FromHF(hf_train, train_transform)
train_fixed = CIFAR10FromHF(hf_train, eval_transform)


In [ ]:
permutation = torch.randperm(
    len(train_augmented),
    generator=torch.Generator().manual_seed(SEED),
).tolist()

train_indices = permutation[:40000]
val_indices = permutation[40000:45000]

train_dataset = Subset(train_augmented, train_indices)
train_eval_dataset = Subset(train_fixed, train_indices)
val_dataset = Subset(train_fixed, val_indices)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_kwargs,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Steps per epoch:", len(train_loader))


## 2. ViT와 공통 초기값

모델 구조:

```text
32x32 image
→ 4x4 patch embedding
→ 6 Transformer blocks
→ LayerNorm
→ classification head
```

네 optimizer는 같은 초기 parameter에서 시작합니다.


In [ ]:
torch.manual_seed(SEED)
base_model = SmallViT()

INITIAL_STATE = {
    name: value.detach().cpu().clone()
    for name, value in base_model.state_dict().items()
}

parameter_count = sum(
    parameter.numel()
    for parameter in base_model.parameters()
)

print(f"Parameters: {parameter_count / 1e6:.2f} M")


## 3. 네 optimizer 학습

비교 대상:

- SGD + momentum
- AdamW
- Prodigy
- Muon

학습 중 기록:

- gradient norm
- gradient cosine
- gradient variance / noise ratio
- update norm
- update-to-weight ratio
- update cosine
- 초기값으로부터 parameter displacement


In [ ]:
histories = {}
dynamics = {}
gradient_noise = {}

for optimizer_name in OPTIMIZER_NAMES:
    print()
    print("=" * 70)
    print("optimizer:", optimizer_name)
    print("=" * 70)

    history_df, dynamics_df, noise_df = train_one_optimizer(
        run_name=optimizer_name,
        initial_state=INITIAL_STATE,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        epochs=EPOCHS,
        diag_epochs=DIAG_EPOCHS,
        dynamics_every=DYNAMICS_EVERY,
        ckpt_dir=LOCAL_CKPT_DIR,
        csv_dir=CSV_DIR,
        tb_dir=TB_DIR,
        amp_enabled=torch.cuda.is_available(),
    )

    histories[optimizer_name] = history_df
    dynamics[optimizer_name] = dynamics_df
    gradient_noise[optimizer_name] = noise_df


### 최종 성능 비교


In [ ]:
history = pd.concat(histories.values(), ignore_index=True)
history.to_csv(CSV_DIR / "all_history.csv", index=False)

final_table = (
    history
    .groupby("run", as_index=False)
    .tail(1)[
        [
            "run",
            "train_accuracy",
            "val_accuracy",
            "val_loss",
            "seconds",
        ]
    ]
    .sort_values("val_accuracy", ascending=False)
)

final_table


In [ ]:
plt.figure(figsize=(9, 5))

for optimizer_name, frame in histories.items():
    plt.plot(
        frame["epoch"],
        frame["val_accuracy"],
        label=optimizer_name,
    )

plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Optimizer comparison")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 4. Gradient / update dynamics

같은 대표 parameter에서 optimizer가 gradient를 실제 update로 어떻게 바꾸는지 비교합니다.

핵심 질문:

1. gradient 방향은 얼마나 일관적인가?
2. minibatch noise는 얼마나 큰가?
3. 실제 update 방향은 gradient와 어떻게 달라지는가?
4. 각 layer는 초기값에서 얼마나 멀리 이동했는가?


In [ ]:
dynamics_all = pd.concat(dynamics.values(), ignore_index=True)
noise_all = pd.concat(gradient_noise.values(), ignore_index=True)

display(dynamics_all.head())
display(noise_all.head())


In [ ]:
tracked_parameter = "blocks.3.mlp.fc1.weight"

plt.figure(figsize=(9, 5))

for optimizer_name in OPTIMIZER_NAMES:
    frame = dynamics_all[
        (dynamics_all["run"] == optimizer_name)
        & (dynamics_all["parameter"] == tracked_parameter)
    ]

    plt.plot(
        frame["step"],
        frame["update_to_weight"],
        label=optimizer_name,
    )

plt.xlabel("training step")
plt.ylabel("update / weight")
plt.title(tracked_parameter)
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 5. Representation geometry

진단 checkpoint 0 / 10 / 25 / 50에서 다음을 계산합니다.

각 layer:

- covariance eigenspectrum
- effective rank
- CKA to initialization
- linear probe accuracy

penultimate representation:

- Neural Collapse NC1 / NC2 / NC3
- margin distribution
- kNN neighborhood purity
- class radius
- class participation dimension
- class-center correlation


In [ ]:
def make_fixed_loader(dataset, sample_count):
    count = min(sample_count, len(dataset))

    subset = Subset(
        dataset,
        list(range(count)),
    )

    return DataLoader(
        subset,
        batch_size=512,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )


rep_train_loader = make_fixed_loader(
    train_eval_dataset,
    REP_TRAIN_SAMPLES,
)

rep_val_loader = make_fixed_loader(
    val_dataset,
    REP_VAL_SAMPLES,
)


In [ ]:
initial_model = SmallViT().to(DEVICE)
initial_model.load_state_dict(INITIAL_STATE)

init_train_features, _, init_train_labels = extract_features_and_logits(
    initial_model,
    rep_train_loader,
    DEVICE,
)

init_val_features, init_val_logits, init_val_labels = extract_features_and_logits(
    initial_model,
    rep_val_loader,
    DEVICE,
)

del initial_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
representation_df, covariance_spectrum_df, class_geometry_df = (
    representation_diagnostics(
        optimizer_names=OPTIMIZER_NAMES,
        diag_epochs=DIAG_EPOCHS,
        initial_state=INITIAL_STATE,
        init_train_features=init_train_features,
        init_train_labels=init_train_labels,
        init_val_features=init_val_features,
        init_val_logits=init_val_logits,
        init_val_labels=init_val_labels,
        rep_train_loader=rep_train_loader,
        rep_val_loader=rep_val_loader,
        ckpt_dir=LOCAL_CKPT_DIR,
        csv_dir=CSV_DIR,
        device=DEVICE,
    )
)

display(representation_df.head(20))


### Class geometry 비교


In [ ]:
class_geometry_df.sort_values(
    ["epoch", "run"],
)[
    [
        "run",
        "epoch",
        "nc1_within_between",
        "nc2_etf_error",
        "nc3_classifier_alignment",
        "margin_mean",
        "margin_p10",
        "knn_purity",
        "mean_class_radius",
        "mean_class_participation_dim",
        "class_center_abs_correlation",
    ]
]


## 6. Function-space 비교

Parameter와 representation이 달라도 최종 함수가 비슷할 수 있습니다.

최종 checkpoint에서 다음을 비교합니다.

- ECE
- pairwise logit MSE
- prediction disagreement


In [ ]:
function_df, function_pair_df = run_function_space_comparison(
    optimizer_names=OPTIMIZER_NAMES,
    final_epoch=EPOCHS,
    loader=rep_val_loader,
    ckpt_dir=LOCAL_CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
)

display(function_df)
display(function_pair_df)


## 7. Hessian Ritz spectrum

기존의 Hessian 최대 고유값 하나만 보는 진단을 Lanczos Ritz spectrum으로 확장합니다.

확인할 것:

- 최대 Ritz value
- 최소 Ritz value
- spectral spread
- gradient와 최대 곡률 방향의 alignment
- gradient와 최소 곡률 방향의 alignment

음의 Ritz value가 잡히면 조사한 부분공간 안에 음의 곡률 방향이 존재한다는 신호로 해석할 수 있습니다.


In [ ]:
hessian_loader = DataLoader(
    Subset(
        val_dataset,
        list(range(HESSIAN_BATCH_SIZE)),
    ),
    batch_size=HESSIAN_BATCH_SIZE,
    shuffle=False,
)

hessian_batch = next(iter(hessian_loader))


In [ ]:
hessian_df, hessian_summary = run_hessian_diagnostics(
    optimizer_names=OPTIMIZER_NAMES,
    final_epoch=EPOCHS,
    initial_state=INITIAL_STATE,
    hessian_batch=hessian_batch,
    ckpt_dir=LOCAL_CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
    steps=LANCZOS_STEPS,
)

hessian_summary


## 8. Mode connectivity / basin

두 optimizer의 최종 checkpoint 사이를 alpha = 0에서 1까지 직선으로 연결합니다.

각 alpha에서 validation loss를 측정하고 다음을 정량화합니다.

- barrier height
- midpoint loss
- midpoint accuracy

단순히 interpolation 그림만 보는 것보다 두 해 사이의 저손실 연결성을 비교하기 쉽습니다.


In [ ]:
connectivity_loader = make_fixed_loader(
    val_dataset,
    CONNECTIVITY_SAMPLES,
)

connectivity_df, barrier_df = run_mode_connectivity(
    optimizer_names=OPTIMIZER_NAMES,
    final_epoch=EPOCHS,
    loader=connectivity_loader,
    ckpt_dir=LOCAL_CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
    max_samples=CONNECTIVITY_SAMPLES,
)

barrier_df


## 9. 최종 해석 순서

결과는 다음 순서로 읽습니다.

```text
성능이 실제로 다른가?
↓
gradient와 update가 어떻게 달랐는가?
↓
어느 layer의 representation이 달라졌는가?
↓
class geometry와 margin이 어떻게 달라졌는가?
↓
실제 출력 함수도 달라졌는가?
↓
최종점 주변의 curvature는 어떻게 다른가?
↓
서로 다른 optimizer의 해 사이에 loss barrier가 있는가?
```

이 구조를 사용하면 단순히 "어느 optimizer가 점수가 높았다"에서 끝나지 않고, 학습동역학 → 표현 → 함수 → loss landscape로 원인을 단계적으로 좁힐 수 있습니다.


In [ ]:
summary = {
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizers": OPTIMIZER_NAMES,
    "final_metrics": final_table.to_dict(orient="records"),
    "hessian": hessian_summary.to_dict(orient="records"),
    "mode_connectivity": barrier_df.to_dict(orient="records"),
}

with open(
    SUMMARY_DIR / "experiment_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved to:", DRIVE_ROOT)
